
# Oxford Bioinformatics Submission — Priors (PNG‑only, Structured)

This notebook is a **computationally light**, **PNG‑only** workflow that reads your **fixed CSVs**,
computes **exhaustive metrics** suitable for a submission package, and **saves figures and metrics
to separate folders**.

**Outputs**  
- **Figures (PNGs only):** `...\results\priors\figure`  
- **Metrics (CSV/JSON/TXT):** `...\results\priors\metric`

**Inputs (hard‑coded):**  
- `detail_global_timeseries.csv`  
- `eb_population_prior.csv`  
- `priors_full_detail.csv`  
- `priors_hyperparams.csv`

> No modeling is performed; this is a plotting/metrics pack over your precomputed outputs.



## 1. Paths & Configuration
Set hard‑coded Windows paths and runtime configuration. Adjust knobs here if needed.


In [1]:

from pathlib import Path

# --- Hard-coded Windows base path ---
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS_DIR = BASE / "results" / "priors"

# Output folders (singular names as requested)
FIGURE_DIR = PRIORS_DIR / "figure"   # PNG figures here
METRIC_DIR = PRIORS_DIR / "metric"   # metrics here
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

# Fixed input files (names do not change)
FILES = {
    "detail_global_timeseries": PRIORS_DIR / "detail_global_timeseries.csv",
    "eb_population_prior":      PRIORS_DIR / "eb_population_prior.csv",
    "priors_full_detail":       PRIORS_DIR / "priors_full_detail.csv",
    "priors_hyperparams":       PRIORS_DIR / "priors_hyperparams.csv",
}

# Runtime knobs
PNG_DPI = 170                    # PNG resolution
MAX_NUMERIC_HISTS = 40           # per dataset
MAX_HIST_SAMPLES = 750_000       # sample cap for histograms
RESAMPLE_RULE = "W"              # weekly resampling for time series
MAX_HEATMAP_NUM_COLS = 60        # correlation heatmap cap
TOP_N_SERIES = 12                # top categories for split series
PAIRWISE_SCATTER_NUM = 6         # top numeric columns (by variance) for pairwise scatter
ACF_LAGS = 24                    # ACF lags (keep modest)

print("BASE:", BASE)
print("Figures  ->", FIGURE_DIR)
print("Metrics  ->", METRIC_DIR)
for k, v in FILES.items():
    print(f"{k:>28s} :", v)


BASE: C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting
Figures  -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figure
Metrics  -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric
    detail_global_timeseries : C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\detail_global_timeseries.csv
         eb_population_prior : C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\eb_population_prior.csv
          priors_full_detail : C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\priors_full_detail.csv
          priors_hyperparams : C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\priors_hyperparams.csv



## 2. Imports & Environment Metadata
We avoid seaborn and keep plotting lightweight with Matplotlib. We also store environment
metadata (versions + config) for reproducibility.


In [2]:

import os, sys, json, gc, re, math, platform
from typing import Optional, Dict, List, Tuple
import numpy as np
import pandas as pd

# Matplotlib (save-only backend)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def env_metadata() -> Dict:
    return {
        "python": sys.version.replace("\n"," "),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
        "config": {
            "PNG_DPI": PNG_DPI,
            "MAX_NUMERIC_HISTS": MAX_NUMERIC_HISTS,
            "MAX_HIST_SAMPLES": MAX_HIST_SAMPLES,
            "RESAMPLE_RULE": RESAMPLE_RULE,
            "MAX_HEATMAP_NUM_COLS": MAX_HEATMAP_NUM_COLS,
            "TOP_N_SERIES": TOP_N_SERIES,
            "PAIRWISE_SCATTER_NUM": PAIRWISE_SCATTER_NUM,
            "ACF_LAGS": ACF_LAGS,
        }
    }

# Save environment snapshot
with open(METRIC_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(env_metadata(), f, indent=2)
print("[json]", METRIC_DIR / "environment.json")


[json] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric\environment.json



## 3. I/O Helpers (safe file names, JSON/CSV writers, CSV readers)
Utility functions for safe filenames, saving PNGs/metrics, and memory‑light CSV reading.


In [3]:

# Safe stems for filenames
INVALID_CHARS = re.compile(r'[<>:"/\\|?*\x00-\x1F]')
def safe_stem(s: str, maxlen: int = 180) -> str:
    return INVALID_CHARS.sub("_", str(s)).strip().replace(" ", "_")[:maxlen]

def _json_default(o):
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    return str(o)

def write_json(obj: Dict, stem: str):
    out = METRIC_DIR / f"{safe_stem(stem)}.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    print("[json]", out)
    return out

def write_csv(df: pd.DataFrame, stem: str):
    out = METRIC_DIR / f"{safe_stem(stem)}.csv"
    df.to_csv(out, index=False)
    print("[csv]", out)
    return out

def save_png_current(stem: str):
    out = FIGURE_DIR / f"{safe_stem(stem)}.png"
    plt.gcf().savefig(out, dpi=PNG_DPI, bbox_inches="tight")
    plt.close(plt.gcf())
    print("[png]", out)

def _human_bytes(b: int) -> str:
    units = ["B","KB","MB","GB","TB"]
    f, i = float(b), 0
    while f >= 1024 and i < len(units)-1:
        f /= 1024.0; i += 1
    return f"{f:.2f} {units[i]}"

def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include=["int64","int32"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

def read_csv_fixed(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists():
        print("[WARN] Missing:", path)
        return None
    try:
        hdr = pd.read_csv(path, nrows=0)
        cols = hdr.columns.tolist()
    except Exception as e:
        print("[ERROR] header read failed:", path, e); return None
    parse_dates = ['date'] if 'date' in cols else None
    try:
        df = pd.read_csv(path, parse_dates=parse_dates, low_memory=False)
    except Exception:
        df = pd.read_csv(path, low_memory=False)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return downcast_numeric(df)



## 4. Generic Metrics (numeric, categorical, missingness, outliers, correlations)
Exhaustive but light metrics for each dataset.


In [4]:

def dataset_overview(dfs: Dict[str, pd.DataFrame]):
    rows = []
    for name, df in dfs.items():
        if df is None:
            rows.append({"dataset": name, "rows": 0, "cols": 0, "mem_bytes": 0})
        else:
            rows.append({
                "dataset": name,
                "rows": int(df.shape[0]),
                "cols": int(df.shape[1]),
                "mem_bytes": int(df.memory_usage(deep=True).sum())
            })
    overview = pd.DataFrame(rows)
    overview["mem_human"] = overview["mem_bytes"].apply(_human_bytes)
    write_csv(overview, "ALL_overview")
    return overview

def numeric_summary(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include="number")
    if num.empty: return pd.DataFrame()
    desc = num.describe(percentiles=[0.001,0.01,0.05,0.25,0.5,0.75,0.95,0.99,0.999]).T
    med = num.median()
    mad = (num - med).abs().median()
    q1 = num.quantile(0.25); q3 = num.quantile(0.75)
    iqr = q3 - q1
    skew = num.skew(numeric_only=True)
    kurt = num.kurtosis(numeric_only=True)
    out = desc.assign(median=med, mad=mad, q1=q1, q3=q3, iqr=iqr, skew=skew, kurtosis=kurt)
    out.reset_index(inplace=True); out.rename(columns={"index":"column"}, inplace=True)
    return out

def categorical_top(df: pd.DataFrame, topn=50) -> pd.DataFrame:
    obj = df.select_dtypes(include="object")
    parts = []
    for c in obj.columns:
        vc = obj[c].value_counts(dropna=True).head(topn)
        vc.index = vc.index.astype(str)
        part = vc.rename_axis("value").reset_index(name="count")
        part.insert(0, "column", c)
        parts.append(part)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def missingness_table(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "missing_pct": [float(df[c].isna().mean()) for c in df.columns],
        "dtype": [str(df[c].dtype) for c in df.columns],
        "unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
    }).sort_values("missing_pct", ascending=False)

def outlier_iqr_table(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include="number")
    rows = []
    for c in num.columns:
        s = pd.to_numeric(df[c], errors="coerce").dropna()
        if s.empty: continue
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
        mask = (s < low) | (s > high)
        rows.append({
            "column": c, "q1": float(q1), "q3": float(q3), "iqr": float(iqr),
            "low_thresh": float(low), "high_thresh": float(high),
            "outliers": int(mask.sum()), "outlier_pct": float(mask.mean())
        })
    return pd.DataFrame(rows).sort_values("outlier_pct", ascending=False)

def corr_and_heatmap(df: pd.DataFrame, dataset_name: str):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 2:
        return None, None
    pearson = df[num_cols].corr(numeric_only=True, method="pearson")
    spearman = df[num_cols].corr(numeric_only=True, method="spearman")
    write_csv(pearson.reset_index().rename(columns={"index":"_col"}), f"{dataset_name}_corr_pearson")
    write_csv(spearman.reset_index().rename(columns={"index":"_col"}), f"{dataset_name}_corr_spearman")
    if len(num_cols) <= MAX_HEATMAP_NUM_COLS:
        plt.figure()
        plt.imshow(pearson.values, aspect="auto")
        plt.title(f"{dataset_name}: Pearson correlation heatmap")
        plt.xticks(range(len(num_cols)), num_cols, rotation=90)
        plt.yticks(range(len(num_cols)), num_cols)
        plt.colorbar()
        save_png_current(f"{dataset_name}__corr_heatmap_pearson")
    return True, True



## 5. Time‑Series Helpers (date/value/category detection, ACF)
Helpers to detect common date/value/category columns and compute a simple ACF.


In [5]:

VAL_CANDS = ["af","allele_frequency","frequency","proportion","value","prevalence","rate","count","counts","n","num","observed"]
CAT_CANDS = ["mutation","variant","lineage","pango_lineage","region","country","location","site","site_id","who_region"]

def find_date(df: pd.DataFrame) -> Optional[str]:
    dt_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
    if dt_cols: return dt_cols[0]
    for c in df.columns:
        if "date" in c.lower() or "time" in c.lower(): return c
    return None

def find_val(df: pd.DataFrame) -> Optional[str]:
    lower = {c.lower(): c for c in df.columns}
    for k in VAL_CANDS:
        if k in lower and pd.api.types.is_numeric_dtype(df[lower[k]]):
            return lower[k]
    num_cols = df.select_dtypes(include="number").columns.tolist()
    return num_cols[0] if num_cols else None

def find_cat(df: pd.DataFrame) -> Optional[str]:
    lower = {c.lower(): c for c in df.columns}
    for k in CAT_CANDS:
        if k in lower: return lower[k]
    obj = df.select_dtypes(include="object").columns.tolist()
    return obj[0] if obj else None

def simple_acf(x: np.ndarray, nlags: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 2: return np.zeros(nlags+1)
    x = x - x.mean()
    c = np.correlate(x, x, mode='full')
    mid = c.size // 2
    c = c[mid:mid+nlags+1]
    c /= c[0] if c[0] != 0 else 1.0
    return c



## 6. Domain‑Specific Metrics (priors & timeseries)
Extra summaries tailored to priors (`μ`, `κ`, AF) and global timeseries.


In [6]:

def priors_full_detail_domain(df: pd.DataFrame) -> Dict:
    d = df.copy()
    if "mu" not in d.columns and "mu_t" in d.columns:
        d = d.rename(columns={"mu_t":"mu"})
    if "kappa" not in d.columns and "kappa_t" in d.columns:
        d = d.rename(columns={"kappa_t":"kappa"})
    out = {}
    for col in ["af","mu","kappa"]:
        if col in d.columns:
            s = pd.to_numeric(d[col], errors="coerce")
            out[f"{col}_desc"] = s.describe(percentiles=[0.001,0.01,0.05,0.25,0.5,0.75,0.95,0.99,0.999]).to_dict()
            if col == "kappa":
                kp = s[s>0]
                if not kp.empty:
                    out["log10_kappa_desc"] = np.log10(kp).describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_dict()
    if {"mu","kappa"}.issubset(d.columns):
        m = pd.to_numeric(d["mu"], errors="coerce")
        k = pd.to_numeric(d["kappa"], errors="coerce")
        mask = m.notna() & k.notna()
        if mask.any():
            out["mu_kappa_corr"] = {
                "pearson": float(pd.Series(m[mask]).corr(pd.Series(k[mask]), method="pearson")),
                "spearman": float(pd.Series(m[mask]).corr(pd.Series(k[mask]), method="spearman")),
                "pearson_mu_log10k": float(pd.Series(m[mask]).corr(np.log10(np.clip(k[mask],1e-12,None)), method="pearson"))
            }
    if "af" in d.columns:
        af = pd.to_numeric(d["af"], errors="coerce")
        out["af_out_of_range"] = {"lt0": int((af < 0).sum()), "gt1": int((af > 1).sum()), "total": int(af.shape[0])}
    return out

def detail_global_timeseries_domain(df: pd.DataFrame) -> Dict:
    out = {}
    dcol = find_date(df); vcol = find_val(df); ccol = find_cat(df)
    if not dcol or not vcol: return out
    d = df[[c for c in [dcol, vcol, ccol] if c]].dropna(subset=[dcol, vcol]).copy().sort_values(dcol)
    if ccol:
        agg = d.groupby([ccol, pd.Grouper(key=dcol, freq=RESAMPLE_RULE)])[vcol].mean().reset_index()
        latest = (agg.dropna(subset=[vcol]).sort_values(dcol).groupby(ccol, as_index=False).last().sort_values(vcol, ascending=False))
        out["top_categories_by_latest"] = latest.head(TOP_N_SERIES).to_dict(orient="list")
        trends = []
        for cat in latest.head(TOP_N_SERIES)[ccol]:
            sub = agg[agg[ccol]==cat].dropna(subset=[vcol])
            if sub.empty: continue
            t = (sub[dcol] - sub[dcol].min()).dt.days.to_numpy()
            y = sub[vcol].to_numpy()
            if t.size >= 2:
                A = np.vstack([t, np.ones_like(t)]).T
                slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
                y_hat = slope*t + intercept
                ss_res = ((y - y_hat)**2).sum()
                ss_tot = ((y - y.mean())**2).sum() if y.size>1 else np.nan
                r2 = 1.0 - ss_res/ss_tot if ss_tot and not np.isnan(ss_tot) else np.nan
                trends.append({"category": str(cat), "slope_per_day": float(slope), "r2": float(r2)})
        out["trends_top"] = trends
    else:
        agg = d.groupby(pd.Grouper(key=dcol, freq=RESAMPLE_RULE))[vcol].mean().reset_index()
        out["series_points"] = int(agg.shape[0])
        acf = simple_acf(agg[vcol].to_numpy(), nlags=ACF_LAGS)
        out["acf"] = [float(x) for x in acf]
    return out

def priors_hyperparams_domain(df: pd.DataFrame) -> Dict:
    out = {}
    lower = {c.lower(): c for c in df.columns}
    if "name" in lower and ("value" in lower or "val" in lower):
        vcol = lower.get("value", lower.get("val"))
        tab = df[[lower["name"], vcol]].copy()
        tab[vcol] = pd.to_numeric(tab[vcol], errors="coerce")
        out["n_params"] = int(tab.shape[0])
        out["value_summary"] = tab[vcol].describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_dict()
    return out

def eb_population_prior_domain(df: pd.DataFrame) -> Dict:
    out = {}
    mu_cols = [c for c in df.columns if c.lower() in ("mu","mu_prior","mu0","mu_init")]
    ka_cols = [c for c in df.columns if c.lower() in ("kappa","kappa_prior","kappa0","kappa_init")]
    if mu_cols:
        s = pd.to_numeric(df[mu_cols[0]], errors="coerce")
        out["mu_desc"] = s.describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_dict()
    if ka_cols:
        k = pd.to_numeric(df[ka_cols[0]], errors="coerce")
        out["kappa_desc"] = k.describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_dict()
        kp = k[k>0]
        if not kp.empty:
            out["log10_kappa_desc"] = np.log10(kp).describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_dict()
    return out



## 7. Plotting (PNG‑only, Matplotlib)
Single‑figure plots only; no seaborn; figures saved & closed immediately.


In [7]:

def plot_numeric_hists(name: str, df: pd.DataFrame):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    plotted = 0
    for col in num_cols:
        if plotted >= MAX_NUMERIC_HISTS: break
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        if s.empty: continue
        if s.size > MAX_HIST_SAMPLES:
            s = s.sample(MAX_HIST_SAMPLES, random_state=RANDOM_SEED)
        plt.figure()
        plt.hist(s.values, bins=60)
        plt.title(f"{name}: histogram of {col}")
        plt.xlabel(col); plt.ylabel("count")
        save_png_current(f"{name}__hist__{col}")
        plotted += 1
        # log histogram for skewed positive columns
        if (s > 0).all():
            skew = float(pd.Series(s).skew())
            if skew > 1.0:
                plt.figure()
                plt.hist(np.log10(s.values), bins=60)
                plt.title(f"{name}: log10 histogram of {col}")
                plt.xlabel(f"log10({col})"); plt.ylabel("count")
                save_png_current(f"{name}__log10hist__{col}")
    return plotted

def plot_pairwise_scatter(name: str, df: pd.DataFrame):
    num = df.select_dtypes(include=[np.number])
    if num.shape[1] < 2: return 0
    vars_by_var = num.var().sort_values(ascending=False).head(PAIRWISE_SCATTER_NUM).index.tolist()
    pairs = []
    for i in range(len(vars_by_var)):
        for j in range(i+1, len(vars_by_var)):
            pairs.append((vars_by_var[i], vars_by_var[j]))
    count = 0
    for x, y in pairs:
        samp = df[[x, y]].dropna()
        if samp.shape[0] > 20000:
            samp = samp.sample(20000, random_state=RANDOM_SEED)
        plt.figure()
        plt.scatter(samp[x].to_numpy(), samp[y].to_numpy(), s=6, alpha=0.6)
        plt.title(f"{name}: scatter {x} vs {y}")
        plt.xlabel(x); plt.ylabel(y)
        save_png_current(f"{name}__scatter__{x}__vs__{y}")
        count += 1
    return count

def plot_corr_heatmap(name: str, df: pd.DataFrame):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 2 or len(num_cols) > MAX_HEATMAP_NUM_COLS: return 0
    corr = df[num_cols].corr(numeric_only=True, method="pearson")
    plt.figure()
    plt.imshow(corr.values, aspect="auto")
    plt.title(f"{name}: Pearson correlation heatmap")
    plt.xticks(range(len(num_cols)), num_cols, rotation=90)
    plt.yticks(range(len(num_cols)), num_cols)
    plt.colorbar()
    save_png_current(f"{name}__corr_heatmap_pearson")
    return 1

def plot_timeseries(name: str, df: pd.DataFrame):
    dcol = find_date(df); vcol = find_val(df); ccol = find_cat(df)
    if not dcol or not vcol: return 0
    d = df[[c for c in [dcol, vcol, ccol] if c]].dropna(subset=[dcol, vcol]).copy()
    if d.empty: return 0
    d = d.sort_values(dcol)
    if ccol:
        agg = d.groupby([ccol, pd.Grouper(key=dcol, freq=RESAMPLE_RULE)])[vcol].mean().reset_index()
        latest = (agg.dropna(subset=[vcol]).sort_values(dcol).groupby(ccol, as_index=False).last().sort_values(vcol, ascending=False))
        top = latest.head(TOP_N_SERIES)[ccol].tolist()
        plt.figure()
        for cat in top:
            sub = agg[agg[ccol]==cat].dropna(subset=[vcol])
            if sub.empty: continue
            plt.plot(sub[dcol], sub[vcol], linewidth=1.6, label=str(cat))
        plt.title(f"{name}: {vcol} over time ({RESAMPLE_RULE}) — top {len(top)} {ccol}")
        plt.xlabel("date"); plt.ylabel(vcol)
        plt.legend(ncol=2, fontsize=8)
        plt.gcf().autofmt_xdate()
        save_png_current(f"{name}__timeseries__{vcol}__{RESAMPLE_RULE}__top_{len(top)}")
        # ACF on overall averaged series
        overall = d.groupby(pd.Grouper(key=dcol, freq=RESAMPLE_RULE))[vcol].mean().dropna()
        if overall.size >= 5:
            acf = simple_acf(overall.to_numpy(), nlags=ACF_LAGS)
            plt.figure()
            plt.stem(range(len(acf)), acf, use_line_collection=True)
            plt.title(f"{name}: ACF ({vcol}, {RESAMPLE_RULE})")
            plt.xlabel("lag"); plt.ylabel("acf")
            save_png_current(f"{name}__acf__{vcol}__{RESAMPLE_RULE}")
        return 2
    else:
        s = d.groupby(pd.Grouper(key=dcol, freq=RESAMPLE_RULE))[vcol].mean().dropna()
        if s.size == 0: return 0
        plt.figure()
        plt.plot(s.index, s.values, linewidth=1.8)
        plt.title(f"{name}: {vcol} over time ({RESAMPLE_RULE})")
        plt.xlabel("date"); plt.ylabel(vcol)
        plt.gcf().autofmt_xdate()
        save_png_current(f"{name}__timeseries__{vcol}__{RESAMPLE_RULE}")
        acf = simple_acf(s.to_numpy(), nlags=ACF_LAGS)
        plt.figure()
        plt.stem(range(len(acf)), acf, use_line_collection=True)
        plt.title(f"{name}: ACF ({vcol}, {RESAMPLE_RULE})")
        plt.xlabel("lag"); plt.ylabel("acf")
        save_png_current(f"{name}__acf__{vcol}__{RESAMPLE_RULE}")
        return 2

def priors_full_detail_plots(df: pd.DataFrame):
    d = df.copy()
    if "mu" not in d.columns and "mu_t" in d.columns:
        d = d.rename(columns={"mu_t":"mu"})
    if "kappa" not in d.columns and "kappa_t" in d.columns:
        d = d.rename(columns={"kappa_t":"kappa"})
    # AF / mu / kappa distributions
    if "af" in d.columns:
        s = pd.to_numeric(d["af"], errors="coerce").dropna()
        if not s.empty:
            plt.figure(); plt.hist(s.values, bins=60)
            plt.title("priors_full_detail: AF distribution"); plt.xlabel("af"); plt.ylabel("count")
            save_png_current("priors_full_detail__af_hist")
    if "mu" in d.columns:
        s = pd.to_numeric(d["mu"], errors="coerce").dropna()
        if not s.empty:
            plt.figure(); plt.hist(s.values, bins=60)
            plt.title("priors_full_detail: μ distribution"); plt.xlabel("mu"); plt.ylabel("count")
            save_png_current("priors_full_detail__mu_hist")
    if "kappa" in d.columns:
        k = pd.to_numeric(d["kappa"], errors="coerce").dropna()
        if not k.empty:
            plt.figure(); plt.hist(k.values, bins=60)
            plt.title("priors_full_detail: κ distribution"); plt.xlabel("kappa"); plt.ylabel("count")
            save_png_current("priors_full_detail__kappa_hist")
            kp = k[k>0]
            if not kp.empty:
                plt.figure(); plt.hist(np.log10(kp.values), bins=60)
                plt.title("priors_full_detail: log10 κ distribution"); plt.xlabel("log10(kappa)"); plt.ylabel("count")
                save_png_current("priors_full_detail__log10_kappa_hist")
    # μ vs log10 κ scatter (sampled)
    if {"mu","kappa"}.issubset(d.columns):
        m = pd.to_numeric(d["mu"], errors="coerce")
        k = pd.to_numeric(d["kappa"], errors="coerce")
        mask = m.notna() & k.notna()
        if mask.any():
            m = m[mask]; k = k[mask]
            n = len(m)
            if n > 20000:
                idx = np.random.RandomState(42).choice(n, size=20000, replace=False)
                m = m.iloc[idx]; k = k.iloc[idx]
            plt.figure()
            plt.scatter(m.values, np.log10(np.clip(k.values, 1e-12, None)), s=8, alpha=0.6)
            plt.title("priors_full_detail: μ vs log10 κ (sample)")
            plt.xlabel("mu"); plt.ylabel("log10(kappa)")
            save_png_current("priors_full_detail__mu_vs_log10kappa_scatter")
    # top-10 mutation AF timeseries
    if {"date","mutation","af"}.issubset(d.columns):
        tmp = d[["date","mutation","af"]].dropna()
        top = (tmp.groupby("mutation")["af"].median().sort_values(ascending=False).head(10).index.tolist())
        sub = tmp[tmp["mutation"].isin(top)]
        sub = sub.groupby(["date","mutation"], as_index=False)["af"].median().sort_values(["date","mutation"])
        plt.figure()
        for mut, g in sub.groupby("mutation"):
            plt.plot(g["date"], g["af"], linewidth=1.5, label=str(mut))
        plt.title("AF by date — top 10 mutations (median)")
        plt.xlabel("date"); plt.ylabel("af")
        plt.legend(ncol=2, fontsize=8); plt.gcf().autofmt_xdate()
        save_png_current("priors_full_detail__af_timeseries_top10")



## 8. Main Execution
Reads CSVs, writes metrics, generates figures, and writes a submission README.


In [8]:

def run_all():
    # Read inputs
    dfs = {}
    for nm, p in FILES.items():
        print(f"[read] {p}")
        dfs[nm] = read_csv_fixed(p)

    # Overview (across datasets)
    overview = dataset_overview(dfs)

    # Per‑dataset exhaustive metrics + plots
    submission_summary = {}
    for name, df in dfs.items():
        if df is None:
            submission_summary[name] = {"status":"missing"}
            continue

        # Base metrics
        base = {
            "shape": {"rows": int(df.shape[0]), "cols": int(df.shape[1])},
            "memory_bytes": int(df.memory_usage(deep=True).sum()),
            "memory_human": _human_bytes(int(df.memory_usage(deep=True).sum()))
        }

        # Generic metrics
        num_sum = numeric_summary(df)
        if not num_sum.empty:
            write_csv(num_sum, f"{name}_numeric_summary")

        cat_top = categorical_top(df, topn=50)
        if not cat_top.empty:
            write_csv(cat_top, f"{name}_categorical_top50")

        miss = missingness_table(df)
        write_csv(miss, f"{name}_missingness")

        outliers = outlier_iqr_table(df)
        if not outliers.empty:
            write_csv(outliers, f"{name}_outliers_iqr")

        # Correlations + heatmap
        corr_and_heatmap(df, name)

        # Plots
        plot_numeric_hists(name, df)
        plot_pairwise_scatter(name, df)
        plot_corr_heatmap(name, df)
        plot_timeseries(name, df)

        # Domain‑specific metrics
        if name == "priors_full_detail":
            dom = priors_full_detail_domain(df); write_json(dom, f"{name}_domain_metrics"); priors_full_detail_plots(df)
        elif name == "detail_global_timeseries":
            dom = detail_global_timeseries_domain(df); write_json(dom, f"{name}_domain_metrics")
        elif name == "priors_hyperparams":
            dom = priors_hyperparams_domain(df); write_json(dom, f"{name}_domain_metrics")
        elif name == "eb_population_prior":
            dom = eb_population_prior_domain(df); write_json(dom, f"{name}_domain_metrics")

        submission_summary[name] = {"base": base}

        gc.collect()

    # Master submission summary + README
    write_json(submission_summary, "submission_summary")
    readme = [
        "# Oxford Bioinformatics submission pack (auto-generated)",
        "## Folders",
        f"- Figures (PNG): {FIGURE_DIR}",
        f"- Metrics (CSV/JSON/TXT): {METRIC_DIR}",
        "## Contents",
        "- *_numeric_summary.csv: extended numeric stats incl. robust metrics (MAD, IQR)",
        "- *_categorical_top50.csv: top 50 values for text columns",
        "- *_missingness.csv: missing counts/%",
        "- *_outliers_iqr.csv: IQR-based outlier thresholds + counts",
        "- *_corr_pearson.csv / *_corr_spearman.csv: correlation matrices",
        "- *_domain_metrics.json: dataset-specific summaries (priors, timeseries)",
        "- submission_summary.json: quick reference for the pack",
        "- environment.json: Python + package versions + config",
        "All plots are PNGs in the 'figure' folder; each chart is a single figure with no styling dependencies."
    ]
    with open(METRIC_DIR / "README_submission.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(readme))
    print("[txt]", METRIC_DIR / "README_submission.txt")

print("Ready. Run the next cell to execute the full pipeline.")


Ready. Run the next cell to execute the full pipeline.



## 9. Run
Execute the pipeline below when your CSVs are present.


In [1]:
# === Priors Stage — FAST Metrics & Diagnostics (PNG-only) =====================
# Outputs:
#   metrics -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric
#   figures -> C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figure
#
# Requirements: numpy, pandas, matplotlib; scipy (betabinom, special) for predictive diagnostics.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- optional (we won't rely on seaborn for speed) ----
try:
    import seaborn as sns
    HAS_SNS = True
    sns.set_theme(style="whitegrid", context="notebook")
except Exception:
    HAS_SNS = False

try:
    from scipy.stats import betabinom, kstest
    from scipy.special import gammaln, psi
    HAS_SCIPY = True
except Exception as e:
    HAS_SCIPY = False
    raise ImportError("This cell needs SciPy (scipy.stats.betabinom, scipy.special). "
                      "Install via: pip install scipy") from e

# ---------------- Paths (hard-coded) ----------------
BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
METRIC = PRIORS / "metrxx435xic"
FIGURE = PRIORS / "figure"
METRIC.mkdir(parents=True, exist_ok=True)
FIGURE.mkdir(parents=True, exist_ok=True)

CSV_PRIORS_FULL = PRIORS / "priors_full_detail.csv"   # expects mu_t/kappa_t/count/coverage/(date)
OUT_PREFIX = "priors"

# ---------------- Speed knobs (tune here) ----------------
RNG = np.random.default_rng(42)

MAX_ROWS_READ          = None      # None = read all; int to cap rows read
ROW_SAMPLE_GLOBAL      = 300_000   # cap rows used in most heavy calcs
ROW_SAMPLE_PIT         = 120_000   # PIT + KS (heavy: cdf)
ROW_SAMPLE_COVERAGE    = 120_000   # coverage ppf (heavy)
ROW_SAMPLE_KL          = 150_000   # info-gain KL
SCATTER_SAMPLE         = 30_000    # scatter plots
HIST_SAMPLE            = 400_000   # histograms
CORR_MAX_NUM_COLS      = 40        # avoid big heatmaps (not used here)
BINS_HIST              = 60        # modest bins
PIT_BINS               = 20
COVERAGE_LEVELS        = (0.5, 0.9, 0.95)

# ---------------- Helpers ----------------
def _save_csv(df: pd.DataFrame, name: str):
    out = METRIC / f"{name}.csv"
    df.to_csv(out, index=False)
    print("[csv]", out)
    return out

def _save_json(obj: dict, name: str):
    import json
    out = METRIC / f"{name}.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    print("[json]", out)
    return out

def _save_png(fig: plt.Figure, name: str, dpi: int = 160):
    out = FIGURE / f"{name}.png"
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print("[png]", out)
    return out

def _clip01(x, eps=1e-9):
    return np.clip(x, eps, 1.0 - eps)

def ab_from_mu_kappa(mu, kappa):
    mu = _clip01(mu, 1e-9)
    kappa = np.clip(kappa, 1e-6, np.inf)
    a = mu * kappa
    b = (1 - mu) * kappa
    return a, b

def log_predictive_density(y, n, a, b):
    # log PMF Beta–Binomial under (a,b), vectorized
    return (
        gammaln(n + 1) - gammaln(y + 1) - gammaln(n - y + 1)
        + gammaln(y + a) + gammaln(n - y + b) - gammaln(n + a + b)
        - (gammaln(a) + gammaln(b) - gammaln(a + b))
    )

def kl_beta(a1, b1, a0, b0):
    # KL(Beta(a1,b1) || Beta(a0,b0)), vectorized
    return (
        (gammaln(a0) + gammaln(b0) - gammaln(a0 + b0))
        - (gammaln(a1) + gammaln(b1) - gammaln(a1 + b1))
        + (a1 - a0) * (psi(a1) - psi(a1 + b1))
        + (b1 - b0) * (psi(b1) - psi(a1 + b1))
    )

def randomized_pit(y, n, a, b, rng: np.random.Generator):
    # U ~ Uniform(F(y-1), F(y)) for discrete cdf
    Fy   = betabinom.cdf(y, n, a, b)
    Fym1 = betabinom.cdf(np.maximum(y - 1, 0), n, a, b)
    U = Fym1 + rng.random(y.shape) * np.maximum(Fy - Fym1, 0.0)
    return np.clip(U, 0.0, 1.0)

def take_sample_idx(N, k):
    if k is None or k >= N:
        return np.arange(N)
    return RNG.choice(N, size=k, replace=False)

def safe_read(path: Path, nrows=MAX_ROWS_READ) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path, nrows=nrows, low_memory=False)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

# ---------------- Load & sanitize ----------------
df = safe_read(CSV_PRIORS_FULL, nrows=MAX_ROWS_READ).copy()

# normalize column names if needed
if "mu" not in df.columns and "mu_t" in df.columns:
    df.rename(columns={"mu_t": "mu"}, inplace=True)
if "kappa" not in df.columns and "kappa_t" in df.columns:
    df.rename(columns={"kappa_t": "kappa"}, inplace=True)

# map case-insensitive required columns
req = {"mu", "kappa", "count", "coverage"}
missing = [c for c in req if c not in {x.lower():x for x in df.columns}.keys()]
if missing:
    lower = {c.lower(): c for c in df.columns}
    for need in list(missing):
        if need in lower:
            df.rename(columns={lower[need]: need}, inplace=True)

for c in ["mu","kappa","count","coverage"]:
    if c not in df.columns:
        raise ValueError(f"priors_full_detail.csv is missing required column: {c}")

# numeric cast + clip
for c in ["mu","kappa","count","coverage"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["mu","kappa","count","coverage"])
df["mu"] = _clip01(df["mu"], 1e-9)
df["kappa"] = np.clip(df["kappa"], 1e-6, np.inf)
df["count"] = df["count"].astype(int)
df["coverage"] = df["coverage"].astype(int)

# optional global row cap for speed
if ROW_SAMPLE_GLOBAL is not None and len(df) > ROW_SAMPLE_GLOBAL:
    df = df.iloc[take_sample_idx(len(df), ROW_SAMPLE_GLOBAL)]

y = df["count"].to_numpy()
n = df["coverage"].to_numpy()
mu = df["mu"].to_numpy()
k  = df["kappa"].to_numpy()
a, b = ab_from_mu_kappa(mu, k)

# ---------------- FAST 1) Coverage calibration (vectorized on a sample) -------
idx_cov = take_sample_idx(len(df), ROW_SAMPLE_COVERAGE)
y_c, n_c, a_c, b_c = y[idx_cov], n[idx_cov], a[idx_cov], b[idx_cov]

rows = []
for lvl in COVERAGE_LEVELS:
    lo = betabinom.ppf((1.0 - lvl)/2.0, n_c, a_c, b_c)
    hi = betabinom.ppf(1.0 - (1.0 - lvl)/2.0, n_c, a_c, b_c)
    inside = (y_c >= lo) & (y_c <= hi)
    rows.append({"nominal": lvl, "empirical": float(np.mean(inside)), "sample_n": int(len(y_c))})
cov_tbl = pd.DataFrame(rows)
_save_csv(cov_tbl, f"{OUT_PREFIX}_predictive_coverage_fast")

fig = plt.figure()
x = cov_tbl["nominal"].to_numpy()
plt.plot(x, cov_tbl["empirical"], marker="o")
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.xlabel("Nominal coverage")
plt.ylabel("Empirical coverage")
plt.title("Predictive coverage (sampled, fast)")
_save_png(fig, f"{OUT_PREFIX}__coverage_nominal_vs_empirical_fast")

# ---------------- FAST 2) PIT uniformity on a sample --------------------------
idx_pit = take_sample_idx(len(df), ROW_SAMPLE_PIT)
y_p, n_p, a_p, b_p = y[idx_pit], n[idx_pit], a[idx_pit], b[idx_pit]
U = randomized_pit(y_p, n_p, a_p, b_p, RNG)

hist, edges = np.histogram(U, bins=PIT_BINS, range=(0,1))
pit_tbl = pd.DataFrame({"bin_left": edges[:-1], "bin_right": edges[1:], "count": hist, "sample_n": int(len(U))})
_save_csv(pit_tbl, f"{OUT_PREFIX}_pit_uniformity_fast")

ks = kstest(U, "uniform")
_save_json({"ks_stat": float(ks.statistic), "ks_pvalue": float(ks.pvalue), "sample_n": int(len(U))},
           f"{OUT_PREFIX}_pit_ks_fast")

fig = plt.figure()
plt.hist(U, bins=PIT_BINS, range=(0,1))
plt.title(f"PIT histogram (fast) — KS p={ks.pvalue:.3g}")
plt.xlabel("u"); plt.ylabel("count")
_save_png(fig, f"{OUT_PREFIX}__pit_hist_fast")

# ---------------- FAST 3) Variance ratio & LPD (fully vectorized) ------------
# model variance: n*mu*(1-mu) * ( (a+b+n)/(a+b+1) )
var_model = n * mu * (1 - mu) * ((a + b + n) / (a + b + 1.0))
resid2 = (y - n * mu)**2
vr_vals = resid2 / np.maximum(var_model, 1e-12)
vr_mean = float(np.mean(vr_vals[np.isfinite(vr_vals)]))

# LPD on a sample (fast)
idx_lpd = take_sample_idx(len(df), min(ROW_SAMPLE_GLOBAL or len(df), 200_000))
lpd_mean = float(np.mean(log_predictive_density(y[idx_lpd], n[idx_lpd], a[idx_lpd], b[idx_lpd])))

_save_json({"variance_ratio_mean": vr_mean,
            "avg_log_predictive_density": lpd_mean},
           f"{OUT_PREFIX}_fit_diagnostics_fast")

fig = plt.figure()
vals = vr_vals
if len(vals) > HIST_SAMPLE:
    vals = vals[take_sample_idx(len(vals), HIST_SAMPLE)]
vals = vals[np.isfinite(vals)]
plt.hist(vals, bins=BINS_HIST)
plt.axvline(1.0, color="k", linestyle="--", linewidth=1)
plt.title("Variance ratio (empirical/model) — fast")
plt.xlabel("ratio"); plt.ylabel("count")
_save_png(fig, f"{OUT_PREFIX}__variance_ratio_hist_fast")

# ---------------- FAST 4) Information gain KL on a sample ---------------------
idx_kl = take_sample_idx(len(df), ROW_SAMPLE_KL)
ap = a[idx_kl] + y[idx_kl]
bp = b[idx_kl] + n[idx_kl] - y[idx_kl]
kl = kl_beta(ap, bp, a[idx_kl], b[idx_kl])
kl_tbl = pd.DataFrame({"kl_beta_post_prior": kl})
_save_csv(kl_tbl, f"{OUT_PREFIX}_information_gain_kl_fast")

fig = plt.figure()
v = kl
if len(v) > HIST_SAMPLE:
    v = v[take_sample_idx(len(v), HIST_SAMPLE)]
plt.hist(v[np.isfinite(v)], bins=BINS_HIST)
plt.title("KL(post || prior) — fast sample")
plt.xlabel("KL"); plt.ylabel("count")
_save_png(fig, f"{OUT_PREFIX}__kl_hist_fast")

# ---------------- FAST 5) Shrinkage (kappa) summary ---------------------------
# (full vectorized, cheap)
kappa_summary = pd.Series(k).describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]).to_frame("kappa").reset_index().rename(columns={"index":"stat"})
_save_csv(kappa_summary, f"{OUT_PREFIX}_shrinkage_kappa_summary_fast")

fig = plt.figure()
kk = k
if len(kk) > HIST_SAMPLE:
    kk = kk[take_sample_idx(len(kk), HIST_SAMPLE)]
plt.hist(kk[np.isfinite(kk)], bins=BINS_HIST)
plt.title("κ (kappa) distribution — fast")
plt.xlabel("kappa"); plt.ylabel("count")
_save_png(fig, f"{OUT_PREFIX}__kappa_hist_fast")

# ---------------- FAST 6) Temporal drift (weekly medians/IQR) -----------------
if "date" in df.columns and pd.api.types.is_datetime64_any_dtype(df["date"]):
    # For speed, aggregate directly; no per-row loops
    d = df.dropna(subset=["date"])[["date","mu","kappa"]].copy()
    d = d.sort_values("date")
    # downsample days by grouping to weeks for fast trend
    grp = d.groupby(pd.Grouper(key="date", freq="W"))
    band = grp.agg(mu_med=("mu","median"),
                   mu_q25=("mu", lambda x: np.quantile(x,0.25)),
                   mu_q75=("mu", lambda x: np.quantile(x,0.75)),
                   k_med=("kappa","median"),
                   k_q25=("kappa", lambda x: np.quantile(x,0.25)),
                   k_q75=("kappa", lambda x: np.quantile(x,0.75))).reset_index()
    _save_csv(band, f"{OUT_PREFIX}_temporal_weekly_median_iqr_fast")

    def linfit(x, y):
        x = (x - x.min()).dt.days.to_numpy(dtype=float)
        y = y.to_numpy(dtype=float)
        if x.size < 2: return np.nan, np.nan
        X = np.c_[x, np.ones_like(x)]
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        yhat = X @ beta
        ssr = np.sum((y - yhat)**2)
        sst = np.sum((y - y.mean())**2) if y.size > 1 else np.nan
        r2 = 1.0 - ssr/sst if (sst and np.isfinite(sst) and sst>0) else np.nan
        return float(beta[0]), float(r2)

    slope_mu, r2_mu = linfit(band["date"], band["mu_med"])
    slope_k , r2_k  = linfit(band["date"], band["k_med"])
    _save_json({"mu_weekly_slope_per_day": slope_mu, "mu_weekly_r2": r2_mu,
                "kappa_weekly_slope_per_day": slope_k, "kappa_weekly_r2": r2_k},
               f"{OUT_PREFIX}_temporal_drift_fast")

    # Plots with IQR ribbon (fast)
    for base, title in [("mu","μ"), ("kappa","κ")]:
        m = band[f"{'mu' if base=='mu' else 'k'}_med"] if base=="mu" else band["k_med"]
        q25 = band[f"{'mu' if base=='mu' else 'k'}_q25"] if base=="mu" else band["k_q25"]
        q75 = band[f"{'mu' if base=='mu' else 'k'}_q75"] if base=="mu" else band["k_q75"]
        fig = plt.figure(figsize=(10,5))
        plt.plot(band["date"], m, label="median")
        plt.fill_between(band["date"], q25, q75, alpha=0.25, label="IQR")
        plt.title(f"{title} weekly median + IQR (fast)")
        plt.xlabel("date"); plt.ylabel(base)
        plt.legend()
        fig.autofmt_xdate()
        _save_png(fig, f"{OUT_PREFIX}__{base}_weekly_median_IQR_fast")

# ---------------- FAST 7) Dependence (μ vs log10 κ) ---------------------------
l10k = np.log10(np.clip(k, 1e-12, None))
pear = float(pd.Series(mu).corr(pd.Series(k), method="pearson"))
spear = float(pd.Series(mu).corr(pd.Series(k), method="spearman"))
pear_l10 = float(pd.Series(mu).corr(pd.Series(l10k), method="pearson"))
_save_json({"pearson_mu_kappa": pear, "spearman_mu_kappa": spear,
            "pearson_mu_log10k": pear_l10}, f"{OUT_PREFIX}_mu_kappa_correlations_fast")

# scatter (sampled, no KDE)
idx_sc = take_sample_idx(len(df), SCATTER_SAMPLE)
fig = plt.figure(figsize=(7.5,6))
plt.scatter(mu[idx_sc], l10k[idx_sc], s=8, alpha=0.4)
plt.xlabel("mu"); plt.ylabel("log10(kappa)")
plt.title("μ vs log10 κ — fast sample")
_save_png(fig, f"{OUT_PREFIX}__mu_vs_log10kappa_fast")

# ---------------- FAST 8) Plausibility checks --------------------------------
af_col = None
for c in ["af","AF","allele_frequency","frequency","proportion"]:
    if c in df.columns:
        af_col = c; break

checks = {}
if af_col is not None:
    af = pd.to_numeric(df[af_col], errors="coerce").to_numpy()
    checks["af_lt0"] = int(np.sum(af < 0))
    checks["af_gt1"] = int(np.sum(af > 1))
    checks["af_total"] = int(af.size)
checks["kappa_eq0"] = int(np.sum(k == 0))
checks["kappa_lt0"] = int(np.sum(k < 0))
checks["kappa_total"] = int(k.size)
_save_json(checks, f"{OUT_PREFIX}_plausibility_checks_fast")

print("\n✅ FAST priors diagnostics complete. CSVs in 'metric', PNGs in 'figure'.")


[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric\priors_predictive_coverage_fast.csv
[png] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figure\priors__coverage_nominal_vs_empirical_fast.png
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric\priors_pit_uniformity_fast.csv
[json] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric\priors_pit_ks_fast.json
[png] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figure\priors__pit_hist_fast.png
[json] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric\priors_fit_diagnostics_fast.json
[png] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\figure\priors__variance_ratio_hist_fast.png
[csv] C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\metric\prio

In [ ]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ZIBB Priors Diagnostics — Exhaustive (CSV-only, hard paths, exact by default)
"""
# (script content elided in this header for brevity – full content below)

import os, json, math, time
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path(r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting")
PRIORS = BASE / "results" / "priors"
IN_CSV = PRIORS / "priors_full_detail.csv"
OUTDIR = PRIORS / "mvb5h7hj67j6 i78o9mnewestyetric"
OUTDIR.mkdir(parents=True, exist_ok=True)

LEVELS = [0.50, 0.90, 0.95]
GRID   = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
PIT_BINS   = 20
HIST_BINS  = 60
SEED       = 42
CHUNK      = 300_000
ROW_METRICS = 0
PIT_SAMPLE  = 200_000

C_MIN = 3
ALPHA = 0.05
POWER = 0.95
LOD_USE_MEDIAN_COVERAGE = True

USE_APPROX_THRESHOLD_N = None

try:
    from scipy.stats import betabinom, kstest, chisquare, spearmanr, pearsonr, kendalltau, skew, kurtosis, normaltest
    try:
        from scipy.stats import cramervonmises
    except Exception:
        cramervonmises = None
    try:
        from scipy.special import erfinv as _erfinv
        HAVE_ERFINV = True
    except Exception:
        HAVE_ERFINV = False
except Exception as e:
    raise SystemExit("SciPy >= 1.7 is required. Please `pip install scipy`.") from e

EPS = 1e-12
np.seterr(all="ignore")

def _clip01(x, eps=1e-12): return np.clip(np.asarray(x, float), eps, 1.0 - eps)
def _save(df, name):
    p = OUTDIR / name; df.to_csv(p, index=False, float_format="%.6g"); print("[csv]", p)

def _hist_df(x, bins=60, rng=None, variable="value"):
    counts, edges = np.histogram(x, bins=bins, range=rng)
    widths = edges[1:] - edges[:-1]
    total = counts.sum()
    dens = np.zeros_like(counts, float)
    if total > 0:
        p = counts / total
        with np.errstate(divide="ignore", invalid="ignore"):
            dens = np.where(widths > 0, p / widths, 0.0)
    return pd.DataFrame({"variable": variable, "bin_left": edges[:-1], "bin_right": edges[1:],
                         "count": counts, "density": dens})

def _summarize(name, arr):
    s = pd.Series(arr).describe(percentiles=[0.01,0.05,0.10,0.25,0.5,0.75,0.90,0.95,0.99])
    s = s.rename(index={"count":"count","mean":"mean","std":"std","min":"min","1%":"q01","5%":"q05",
                        "10%":"q10","25%":"q25","50%":"q50","75%":"q75","90%":"q90","95%":"q95","99%":"q99","max":"max"})
    out = s.to_frame("value").reset_index().rename(columns={"index":"stat"})
    out.insert(0,"variable",name); return out

def _wilson(p_hat, n, z=1.96):
    if n <= 0: return (np.nan, np.nan)
    denom = 1 + z**2/n
    center = (p_hat + (z**2)/(2*n)) / denom
    half = z * math.sqrt((p_hat*(1-p_hat)/n) + (z**2)/(4*n**2)) / denom
    return (center - half, center + half)

def _pearson(x, y):
    r = pearsonr(x, y); return float(getattr(r,"statistic",r[0])), float(getattr(r,"pvalue",r[1]))
def _spearman(x, y):
    r = spearmanr(x, y); return float(getattr(r,"correlation",r[0])), float(getattr(r,"pvalue",r[1]))
def _kendall(x, y):
    r = kendalltau(x, y); return float(getattr(r,"correlation",r[0])), float(getattr(r,"pvalue",r[1]))

def _mix_mean_var(n, mu, kappa, pi):
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, np.inf)
    a = mu_c * k_c; b = (1.0 - mu_c) * k_c
    m_bb = n * mu_c
    v_bb = n * mu_c * (1.0 - mu_c) * ((a + b + n) / (a + b + 1.0))
    m_mix = (1.0 - pi) * m_bb
    v_mix = (1.0 - pi) * v_bb + (pi * (1.0 - pi)) * (m_bb ** 2)
    return m_mix, np.maximum(v_mix, EPS)

def _mix_logpmf(y, n, mu, kappa, pi):
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, np.inf)
    a = mu_c * k_c; b = (1.0 - mu_c) * k_c
    y = np.asarray(y, int)
    log_bb = betabinom.logpmf(y, n, a, b)
    log1m  = np.log1p(-np.clip(pi, 0.0, 1.0 - EPS))
    out = log1m + log_bb
    z = (y == 0)
    if np.any(z):
        log_pi  = np.log(np.clip(pi[z], EPS, 1.0 - EPS))
        log_bb0 = betabinom.logpmf(0, n[z], a[z], b[z])
        M = np.vstack([log_pi, log1m[z] + log_bb0])
        amax = np.max(M, axis=0)
        out[z] = amax + np.log(np.exp(M[0] - amax) + np.exp(M[1] - amax))
    return out

def _ppf_exact(q, n, a, b, pi):
    qprime = (q - pi) / np.maximum(1.0 - pi, EPS)
    qprime = np.clip(qprime, 0.0, 1.0 - EPS)
    y = betabinom.ppf(qprime, n, a, b).astype(np.int32)
    y = np.where(q <= pi, 0, y)
    return np.clip(y, 0, n.astype(np.int32))

def _ppf_norm(q, n, a, b, pi):
    mu_s = a / np.maximum(a + b, EPS)
    v_bb = n * mu_s * (1.0 - mu_s) * ((n + a + b) / (a + b + 1.0))
    m_bb = n * mu_s
    m = (1.0 - pi) * m_bb
    v = (1.0 - pi) * v_bb + (pi * (1.0 - pi)) * (m_bb ** 2)
    sd = np.sqrt(np.maximum(v, EPS))
    if HAVE_ERFINV:
        z = np.sqrt(2.0) * _erfinv(np.clip(2.0 * np.clip((q - pi) / np.maximum(1 - pi, EPS), 0.0, 1.0) - 1.0, -1.0, 1.0))
    else:
        def erfinv(x):
            a = 0.147; sgn = np.sign(x); ln = np.log(1 - x * x)
            first = 2 / (np.pi * a) + ln / 2; second = ln / a
            return sgn * np.sqrt(np.sqrt(first * first - second) - first)
        z = np.sqrt(2.0) * erfinv(np.clip(2.0 * np.clip((q - pi) / np.maximum(1 - pi, EPS), 0.0, 1.0) - 1.0, -1.0, 1.0))
    y = np.floor(m + z * sd + 0.5).astype(np.int32)
    y = np.where(q <= pi, 0, y)
    return np.clip(y, 0, n.astype(np.int32))

def _ppf_chunks(qs, n, mu, kappa, pi, chunk=CHUNK):
    qs = np.atleast_1d(np.asarray(qs, float))
    N = len(n); out = np.empty((qs.size, N), dtype=np.int32)
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, np.inf)
    a = mu_c * k_c; b = (1.0 - mu_c) * k_c; pi_c = np.clip(pi, 0.0, 1.0 - EPS)
    use_ap = np.zeros(N, dtype=bool)
    if isinstance(USE_APPROX_THRESHOLD_N, (int, float)):
        use_ap = (np.asarray(n) >= int(USE_APPROX_THRESHOLD_N))
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        n_s, a_s, b_s, pi_s = n[s:e], a[s:e], b[s:e], pi_c[s:e]
        ap_s = use_ap[s:e]
        for i, q in enumerate(qs):
            if ap_s.any():
                y = np.empty(e - s, dtype=np.int32)
                if (~ap_s).any(): y[~ap_s] = _ppf_exact(q, n_s[~ap_s], a_s[~ap_s], b_s[~ap_s], pi_s[~ap_s])
                y[ap_s] = _ppf_norm(q, n_s[ap_s], a_s[ap_s], b_s[ap_s], pi_s[ap_s])
            else:
                y = _ppf_exact(q, n_s, a_s, b_s, pi_s)
            out[i, s:e] = y
    return out[0] if qs.size == 1 else out

def _cdf_exact(y, n, a, b, pi):
    F_bb = betabinom.cdf(np.clip(y, -1, None), n, a, b)
    return np.clip(pi + (1.0 - pi) * F_bb, 0.0, 1.0)

def _cdf_norm(y, n, a, b, pi):
    from math import erf, sqrt
    mu_s = a / np.maximum(a + b, EPS)
    v_bb = n * mu_s * (1.0 - mu_s) * ((n + a + b) / (a + b + 1.0))
    m_bb = n * mu_s
    m = (1.0 - pi) * m_bb
    v = (1.0 - pi) * v_bb + (pi * (1.0 - pi)) * (m_bb ** 2)
    sd = np.sqrt(np.maximum(v, EPS))
    z = (y + 0.5 - m) / sd
    Phi = 0.5 * (1.0 + erf(z / sqrt(2.0)))
    return np.clip(pi + (1.0 - pi) * np.clip(Phi, 0.0, 1.0), 0.0, 1.0)

def _cdf_chunks(y_vals, n, mu, kappa, pi, chunk=CHUNK):
    N = len(n); out = np.empty(N, dtype=np.float64)
    mu_c = _clip01(mu); k_c = np.clip(kappa, EPS, np.inf)
    a = mu_c * k_c; b = (1.0 - mu_c) * k_c; pi_c = np.clip(pi, 0.0, 1.0)
    use_ap = np.zeros(N, dtype=bool)
    if isinstance(USE_APPROX_THRESHOLD_N, (int, float)):
        use_ap = (np.asarray(n) >= int(USE_APPROX_THRESHOLD_N))
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        n_s, a_s, b_s, pi_s = n[s:e], a[s:e], b[s:e], pi_c[s:e]
        y_s = y_vals[s:e]; ap_s = use_ap[s:e]
        if ap_s.any():
            out[s:e] = 0.0
            if (~ap_s).any(): out[s:e][~ap_s] = _cdf_exact(y_s[~ap_s], n_s[~ap_s], a_s[~ap_s], b_s[~ap_s], pi_s[~ap_s])
            out[s:e][ap_s] = _cdf_norm(y_s[ap_s], n_s[ap_s], a_s[ap_s], b_s[ap_s], pi_s[ap_s])
        else:
            out[s:e] = _cdf_exact(y_s, n_s, a_s, b_s, pi_s)
    return np.clip(out, 0.0, 1.0)

def _roc_pr_from_scores(y_true, scores):
    order = np.argsort(scores)
    s = scores[order]; t = y_true[order]
    uniq = np.unique(s)
    if uniq.size > 400:
        qs = np.quantile(s, np.linspace(0, 1, 401))
        thr = np.unique(qs)
    else:
        thr = uniq
    TPR=[]; FPR=[]; PREC=[]; REC=[]
    P = float(np.sum(t==1)); N0 = float(np.sum(t==0))
    if P==0 or N0==0:
        return pd.DataFrame(columns=["threshold","TPR","FPR","precision","recall"]), np.nan, np.nan
    for th in thr[::-1]:
        pred = (scores >= th)
        TP = float(np.sum(pred & (y_true==1)))
        FP = float(np.sum(pred & (y_true==0)))
        FN = P - TP; TN = N0 - FP
        tpr = TP / P; fpr = FP / N0
        prec = TP / max(TP+FP, 1.0); rec = tpr
        TPR.append(tpr); FPR.append(fpr); PREC.append(prec); REC.append(rec)
    auc = 0.0
    if len(FPR) > 1:
        x = np.array(FPR); y = np.array(TPR)
        order = np.argsort(x); auc = float(np.trapz(y[order], x[order]))
    ap = 0.0
    if len(REC) > 1:
        x = np.array(REC); y = np.array(PREC)
        order = np.argsort(x); ap = float(np.trapz(y[order], x[order]))
    curve = pd.DataFrame({"threshold":thr[::-1],"TPR":TPR,"FPR":FPR,"precision":PREC,"recall":REC})
    return curve, auc, ap

t0 = time.time()
if not IN_CSV.exists(): raise FileNotFoundError(f"Missing priors CSV: {IN_CSV}")

df = pd.read_csv(IN_CSV, low_memory=False)
if "mu" in df.columns and "mu_t" not in df.columns: df.rename(columns={"mu":"mu_t"}, inplace=True)
if "kappa" in df.columns and "kappa_t" not in df.columns: df.rename(columns={"kappa":"kappa_t"}, inplace=True)
if "pi" not in df.columns: df["pi"] = 0.0

for c in ["count","coverage","mu_t","kappa_t","pi"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["count","coverage","mu_t","kappa_t","pi"]).copy()
df["count"]    = df["count"].astype(int)
df["coverage"] = df["coverage"].astype(int)
df["count"] = np.clip(df["count"], 0, df["coverage"])
df["mu_t"]    = _clip01(df["mu_t"], 1e-12)
df["kappa_t"] = np.clip(df["kappa_t"], EPS, 1e12)
df["pi"]      = np.clip(df["pi"], 0.0, 1.0 - 1e-12)

y  = df["count"].to_numpy(int)
n  = df["coverage"].to_numpy(int)
mu = df["mu_t"].to_numpy(float)
k  = df["kappa_t"].to_numpy(float)
pi = df["pi"].to_numpy(float)
N  = len(df)
print(f"[load] rows={N:,} | priors={IN_CSV}")

mean,var = _mix_mean_var(n,mu,k,pi)
sd = np.sqrt(var)
resid = y - mean
with np.errstate(divide="ignore", invalid="ignore"):
    z = resid / sd
logpmf = _mix_logpmf(y,n,mu,k,pi)

# Coverage
cov_rows=[]; width_rows=[]
qs = np.asarray(LEVELS, float); qlo = (1.0 - qs)/2.0; qhi = (1.0 + qs)/2.0
los = _ppf_chunks(qlo, n, mu, k, pi, CHUNK); his = _ppf_chunks(qhi, n, mu, k, pi, CHUNK)
for i, lvl in enumerate(qs):
    lo, hi = los[i], his[i]
    inside = (y >= lo) & (y <= hi); lower=(y<lo); upper=(y>hi)
    emp=float(inside.mean()); wil_lo,wil_hi=_wilson(emp,N)
    cov_rows.append({"nominal":float(lvl),"empirical":emp,"bias":emp-float(lvl),
                     "lower_miss_rate":float(lower.mean()),"upper_miss_rate":float(upper.mean()),
                     "asymmetry_upper_minus_lower":float(upper.mean()-lower.mean()),
                     "mean_width":float((hi-lo).mean()),"median_width":float(np.median(hi-lo)),
                     "wilson_lo":wil_lo,"wilson_hi":wil_hi,"sample_n":int(N)})
    width_rows.append({"level":float(lvl),"mean_width":float((hi-lo).mean()),"median_width":float(np.median(hi-lo))})
_save(pd.DataFrame(cov_rows), "priors_predictive_coverage.csv")
_save(pd.DataFrame(width_rows), "priors_width_by_level.csv")

grid = np.asarray(GRID, float)
glo = (1.0 - grid)/2.0; ghi = (1.0 + grid)/2.0
g_los = _ppf_chunks(glo, n, mu, k, pi, CHUNK); g_his = _ppf_chunks(ghi, n, mu, k, pi, CHUNK)
grid_rows=[]
for i, lvl in enumerate(grid):
    inside = (y >= g_los[i]) & (y <= g_his[i])
    grid_rows.append({"nominal": float(lvl), "empirical": float(inside.mean()),
                      "bias": float(inside.mean() - lvl), "sample_n": int(N)})
_save(pd.DataFrame(grid_rows), "priors_predictive_coverage_grid.csv")

def coverage_by_bins(values, name, levels):
    labels = pd.qcut(values, q=10, duplicates="drop")
    rows=[]
    for i, lvl in enumerate(levels):
        lo = g_los[i]; hi = g_his[i]
        inside = (y >= lo) & (y <= hi)
        g = pd.DataFrame({name: labels, "inside": inside})
        gg = g.groupby(name, dropna=False)["inside"].mean().reset_index()
        for _, r in gg.iterrows():
            rows.append({"bin": str(r[name]), "nominal": float(lvl), "empirical": float(r["inside"]), "variable": name})
    return pd.DataFrame(rows)

lam = n * mu; l10k = np.log10(np.clip(k, EPS, None))
cov_bins = pd.concat([
    coverage_by_bins(mu, "mu_decile", grid),
    coverage_by_bins(l10k, "l10k_decile", grid),
    coverage_by_bins(lam, "lambda_decile", grid),
], ignore_index=True)
_save(cov_bins, "priors_coverage_by_bins.csv")

# PIT
rng = np.random.default_rng(SEED)
take = np.arange(N) if N <= PIT_SAMPLE else rng.choice(N, PIT_SAMPLE, replace=False)
F_y  = _cdf_chunks(y[take],   n[take], mu[take], k[take], pi[take], CHUNK)
F_ym = _cdf_chunks(y[take]-1, n[take], mu[take], k[take], pi[take], CHUNK)
U = np.clip(F_ym + rng.random(take.size) * np.maximum(F_y - F_ym, 0.0), 0.0, 1.0)

hist,edges=np.histogram(U,bins=PIT_BINS,range=(0,1))
expected=take.size/PIT_BINS
pit_bins = pd.DataFrame({"bin_left":edges[:-1],"bin_right":edges[1:],
                         "count":hist,"density":hist/max(hist.sum(),1),"expected_count_uniform":expected,
                         "sample_n":take.size})
_save(pit_bins, "priors_pit_uniformity_bins.csv")

ks=kstest(U,"uniform")
chi=chisquare(hist,f_exp=np.full_like(hist,expected,dtype=float))
try:
    cvm=cramervonmises(U,"uniform"); cvm_stat=float(cvm.statistic); cvm_p=float(cvm.pvalue)
except Exception:
    cvm_stat=cvm_p=np.nan
pit_tests = pd.DataFrame([{
    "test":"KS (Uniform)","statistic":float(getattr(ks,"statistic",ks[0])),"pvalue":float(getattr(ks,"pvalue",ks[1])),
    "N":int(take.size),"bins":PIT_BINS
},{
    "test":"Chi-square (Uniform bins)","statistic":float(getattr(chi,"statistic",chi[0])),"pvalue":float(getattr(chi,"pvalue",chi[1])),
    "N":int(take.size),"bins":PIT_BINS
},{
    "test":"Cramér–von Mises (Uniform)","statistic":cvm_stat,"pvalue":cvm_p,"N":int(take.size),"bins":PIT_BINS
}])
_save(pit_tests, "priors_pit_tests.csv")

# Fit diagnostics
with np.errstate(divide="ignore",invalid="ignore"):
    vr=(resid**2)/var
vr_fin=vr[np.isfinite(vr)]
rmsz=float(np.sqrt(np.mean(z**2)))
z_mean=float(np.mean(z)); z_std=float(np.std(z))
z_sk=float(skew(z,nan_policy="omit")); z_ku=float(kurtosis(z,fisher=True,nan_policy="omit"))
try:
    k2=normaltest(z,nan_policy="omit"); k2_stat=float(getattr(k2,"statistic",k2[0])); k2_p=float(getattr(k2,"pvalue",k2[1]))
except Exception:
    k2_stat=k2_p=np.nan

r_p,p_p=_pearson(y,mean); r_s,p_s=_spearman(y,mean)
cov_ym=float(np.cov(mean,y,ddof=0)[0,1]) if N>1 else np.nan
var_m=float(np.var(mean)) if N>0 else np.nan
slope=float(cov_ym/var_m) if var_m and np.isfinite(var_m) and var_m>0 else np.nan
intercept=float(np.mean(y)-slope*np.mean(mean)) if np.isfinite(slope) else np.nan
r2=float(r_p**2) if np.isfinite(r_p) else np.nan

fit_df = pd.DataFrame([{
    "N":int(N),
    "rmse":float(np.sqrt(np.mean(resid**2))), "mae":float(np.mean(np.abs(resid))),
    "rmsz":rmsz, "avg_logpmf":float(np.mean(logpmf[np.isfinite(logpmf)])),
    "variance_ratio_mean":float(np.mean(vr_fin)),
    "variance_ratio_median":float(np.median(vr_fin)),
    "variance_ratio_q05":float(np.quantile(vr_fin,0.05)),
    "variance_ratio_q95":float(np.quantile(vr_fin,0.95)),
    "z_mean":z_mean, "z_std":z_std, "z_skew":z_sk, "z_kurtosis_fisher":z_ku,
    "normaltest_k2_stat":k2_stat, "normaltest_k2_pvalue":k2_p,
    "pearson_y_mean_r":r_p, "pearson_y_mean_p":p_p,
    "spearman_y_mean_rho":r_s, "spearman_p":p_s,
    "regress_y_on_mean_slope":slope, "regress_y_on_mean_intercept":intercept, "regress_y_on_mean_r2":r2
}])
_save(fit_df, "priors_fit_diagnostics.csv")

# Variable summaries
var_summary = pd.concat([
    _summarize("y",y), _summarize("n",n), _summarize("mu",mu), _summarize("kappa",k), _summarize("pi",pi),
    _summarize("mean",mean), _summarize("var",var), _summarize("sd",np.sqrt(var)),
    _summarize("z",z), _summarize("resid",resid), _summarize("u_pit",U),
    _summarize("variance_ratio",vr), _summarize("logpmf",logpmf[np.isfinite(logpmf)]),
], ignore_index=True)
_save(var_summary, "priors_variable_summaries.csv")

# Correlations
r_pk,p_pk=_pearson(mu,k); r_sk,p_sk=_spearman(mu,k); r_kk,p_kk=_kendall(mu,k)
_save(pd.DataFrame([{
    "pearson_mu_kappa":r_pk,"pearson_p":p_pk,
    "spearman_mu_kappa":r_sk,"spearman_p":p_sk,
    "kendall_mu_kappa":r_kk,"kendall_p":p_kk
}]), "priors_mu_kappa_correlation.csv")
_save(pd.DataFrame([{
    "pearson_y_mean":r_p,"pearson_p":p_p,
    "spearman_y_mean":r_s,"spearman_p":p_s,
    "slope_y_on_mean":slope,"intercept_y_on_mean":intercept,"r2":r2
}]), "priors_y_mean_correlation.csv")

# Histograms
z_lo,z_hi = float(np.nanquantile(z,0.001)), float(np.nanquantile(z,0.999))
vr_lo,vr_hi = float(np.nanquantile(vr_fin,0.0)), float(np.nanquantile(vr_fin,0.995))
h = pd.concat([
    _hist_df(mu,bins=HIST_BINS,rng=(0,1),variable="mu"),
    _hist_df(np.log10(np.clip(k, EPS, None)),bins=HIST_BINS,
             rng=(float(np.nanmin(np.log10(np.clip(k, EPS, None)))), float(np.nanmax(np.log10(np.clip(k, EPS, None))))),
             variable="log10_kappa"),
    _hist_df(np.clip(z,z_lo,z_hi),bins=HIST_BINS,rng=(z_lo,z_hi),variable="z"),
    _hist_df(U,bins=HIST_BINS,rng=(0,1),variable="u_pit"),
    _hist_df(np.clip(vr,vr_lo,vr_hi),bins=HIST_BINS,rng=(vr_lo,vr_hi),variable="variance_ratio"),
], ignore_index=True)
_save(h, "priors_histograms.csv")

# Calibration by deciles
def cov_by_quant(values, name, levels):
    labels = pd.qcut(values, q=10, duplicates="drop")
    rows=[]
    for lvl in levels:
        qlo,qhi=(1-lvl)/2,(1+lvl)/2
        lo=_ppf_chunks(qlo, n, mu, k, pi, CHUNK); hi=_ppf_chunks(qhi, n, mu, k, pi, CHUNK)
        inside=(y>=lo)&(y<=hi)
        g=pd.DataFrame({name:labels,"inside":inside})
        gg=g.groupby(name,dropna=False)["inside"].mean().reset_index()
        for _,r in gg.iterrows():
            rows.append({"bin":str(r[name]),"nominal":float(lvl),"empirical":float(r["inside"])})
    return pd.DataFrame(rows)

lam = n*mu
_save(cov_by_quant(mu,"mu_decile",LEVELS), "priors_calibration_by_mu_decile.csv")
_save(cov_by_quant(np.log10(np.clip(k,EPS,None)),"l10k_decile",LEVELS), "priors_calibration_by_l10k_decile.csv")
_save(cov_by_quant(lam,"lambda_decile",LEVELS), "priors_calibration_by_lambda_decile.csv")

# Error by coverage decile
with np.errstate(divide="ignore",invalid="ignore"):
    prop_resid=(y/np.maximum(n,1))-mu
labels_cov = pd.qcut(n, q=10, duplicates="drop")
gg = pd.DataFrame({"cov_decile":labels_cov,"abs_resid_count":np.abs(resid),"abs_resid_prop":np.abs(prop_resid)}) \
        .groupby("cov_decile").agg(abs_resid_count=("abs_resid_count","mean"),
                                   abs_resid_prop=("abs_resid_prop","mean"),
                                   N=("abs_resid_count","size")).reset_index()
_save(gg, "priors_error_by_coverage_decile.csv")

# PIT by site/mutation
if "site_id" in df.columns:
    rows=[]
    tmp=pd.DataFrame({"site_id":df["site_id"].astype(str),"U":U})
    for sid,grp in tmp.groupby("site_id"):
        if len(grp)<10: continue
        r=kstest(grp["U"].to_numpy(float),"uniform")
        rows.append({"site_id":sid,"n":int(len(grp)),"ks_stat":float(getattr(r,"statistic",r[0])),"ks_pvalue":float(getattr(r,"pvalue",r[1]))})
    if rows: _save(pd.DataFrame(rows).sort_values("ks_stat",ascending=False), "priors_pit_ks_by_site.csv")

if "mutation" in df.columns and df["mutation"].nunique() <= 1000:
    rows=[]
    tmpm=pd.DataFrame({"mutation":df["mutation"].astype(str),"U":U})
    for m,grp in tmpm.groupby("mutation"):
        if len(grp)<10: continue
        r=kstest(grp["U"].to_numpy(float),"uniform")
        rows.append({"mutation":m,"n":int(len(grp)),"ks_stat":float(getattr(r,"statistic",r[0])),"ks_pvalue":float(getattr(r,"pvalue",r[1]))})
    if rows: _save(pd.DataFrame(rows).sort_values("ks_stat",ascending=False), "priors_pit_ks_by_mutation.csv")

# Extreme misses (outside 99%)
lo99=_ppf_chunks(0.005, n, mu, k, pi, CHUNK); hi99=_ppf_chunks(0.995, n, mu, k, pi, CHUNK)
miss=(y<lo99)|(y>hi99); idx=np.where(miss)[0]
if idx.size>0:
    take=idx if idx.size<=50000 else np.random.default_rng(123).choice(idx,50000,replace=False)
    miss_df=pd.DataFrame({"y":y[take],"n":n[take],"mu":mu[take],"kappa":k[take],"pi":pi[take],
                          "mean":mean[take],"sd":sd[take],"z":z[take],
                          "y_lo_99":lo99[take],"y_hi_99":hi99[take]})
    for opt in ["site_id","mutation","date"]:
        if opt in df.columns: miss_df[opt]=df[opt].astype(str).values[take]
    _save(miss_df, "priors_outside_99_sample.csv")

# Detection metrics
present = (y >= C_MIN).astype(int)
p_detect = 1.0 - _cdf_chunks(C_MIN - 1, n, mu, k, pi, CHUNK)
curve, auc, ap = _roc_pr_from_scores(present, p_detect)
if not curve.empty: _save(curve, "priors_detection_curve.csv")
brier = float(np.mean((p_detect - present)**2)) if present.size else np.nan
det_summary = pd.DataFrame([{"C_min":C_MIN, "ROC_AUC":auc, "PR_AUC":ap, "Brier":brier,
                             "positive_rate": float(present.mean()), "N": int(N)}])
_save(det_summary, "priors_detection_summary.csv")

# LOD
def lod_mu_for_row(n_i, k_i, pi_i, c_min=C_MIN, power=POWER, tol=1e-4, max_iter=50):
    lo, hi = 1e-6, 0.5
    for _ in range(max_iter):
        mid = 0.5*(lo+hi)
        a = mid * max(k_i, 0.0); b = (1.0 - mid) * max(k_i, 0.0)
        q = betabinom.cdf(c_min-1, n_i, a, b) if n_i>0 else 1.0
        p_ge = (1.0 - pi_i) * (1.0 - q)
        if p_ge >= power: hi = mid
        else: lo = mid
        if (hi - lo) < tol: break
    return hi

rows = []
if "mutation" in df.columns:
    for mut, g in df.groupby("mutation", sort=False):
        n_star = int(np.median(g["coverage"])) if LOD_USE_MEDIAN_COVERAGE else int(np.max(g["coverage"]))
        k_star = float(np.median(g["kappa_t"])); pi_star = float(np.median(g["pi"]))
        lod = lod_mu_for_row(n_star, k_star, pi_star)
        rows.append({"group":"mutation","key":str(mut),"n_star":n_star,"kappa_star":k_star,"pi_star":pi_star,"LOD_mu":lod})
if "site_id" in df.columns:
    for sid, g in df.groupby("site_id", sort=False):
        n_star = int(np.median(g["coverage"])) if LOD_USE_MEDIAN_COVERAGE else int(np.max(g["coverage"]))
        k_star = float(np.median(g["kappa_t"])); pi_star = float(np.median(g["pi"]))
        lod = lod_mu_for_row(n_star, k_star, pi_star)
        rows.append({"group":"site_id","key":str(sid),"n_star":n_star,"kappa_star":k_star,"pi_star":pi_star,"LOD_mu":lod})
if rows: _save(pd.DataFrame(rows), "priors_lod_summaries.csv")

# QA
qa=[]
cov_df=pd.DataFrame(cov_rows)
for _,r in cov_df.iterrows():
    nom=float(r["nominal"]); emp=float(r["empirical"]); bias=float(r["bias"]); asym=float(r["asymmetry_upper_minus_lower"])
    tol=0.02 if nom>=0.9 else 0.03; asym_tol=0.02 if nom>=0.9 else 0.03
    status="PASS" if (abs(bias)<=tol and abs(asym)<=asym_tol) else "MINOR"
    if abs(bias)>tol+0.02 or abs(asym)>asym_tol+0.02: status="FLAG"
    qa.append({"module":"Coverage","metric":f"{nom:.2f}","value":f"emp={emp:.4f}, bias={bias:.4f}, asym={asym:.4f}","status":status})

ks_stat=float(getattr(ks,"statistic",ks[0])); ks_p=float(getattr(ks,"pvalue",ks[1]))
ks_status="PASS" if ks_stat<=0.03 else ("MINOR" if ks_stat<=0.05 else "FLAG")
qa.append({"module":"PIT","metric":"KS","value":f"KS={ks_stat:.4f} (p={ks_p:.2e})","status":ks_status})

vr_mean=float(np.mean(vr_fin)); z_mean_ok=abs(float(np.mean(z)))<=0.05
z_std_ok=0.95<=float(np.std(z))<=1.05; rmsz_ok=0.95<=float(np.sqrt(np.mean(z**2)))<=1.05
vrm_ok=0.9<=vr_mean<=1.1
fit_status="PASS" if all([z_mean_ok,z_std_ok,rmsz_ok,vrm_ok]) else ("MINOR" if sum([z_mean_ok,z_std_ok,rmsz_ok,vrm_ok])>=3 else "FLAG")
qa.append({"module":"Fit","metric":"z/VR","value":f"z_mean={float(np.mean(z)):.3f}, z_std={float(np.std(z)):.3f}, RMSZ={float(np.sqrt(np.mean(z**2))):.3f}, VR_mean={vr_mean:.3f}","status":fit_status})

if not math.isnan(auc):
    det_status="PASS" if auc>=0.85 else ("MINOR" if auc>=0.75 else "FLAG")
    qa.append({"module":"Detection","metric":"ROC-AUC","value":f"auc={auc:.3f}, pr_auc={ap:.3f}, brier={brier:.4f}","status":det_status})

_save(pd.DataFrame(qa), "priors_qa_summary.csv")

print("[done]", json.dumps({"N":int(N), "outdir": OUTDIR.as_posix(), "sec": round(time.time()-t0,2)}))


[load] rows=247,614 | priors=C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\priors\priors_full_detail.csv
